In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# VSM x Mechanism (variant-level non-binary data)
# Main Question: How important is a given mechanism/feature to a given VSM?
# Analysis:
## correlation of the given mechanism with a given VSM per ENSG
## Average coefficient of the given mechanism after fitting a linear regression model for a given VSM per ENSG (vsm_score ~ mechanism)
## VSM score and mechanism converted to within-protein rank

In [3]:
def add_within_gene_ranks(df, vsms, feature):
    for vsm in vsms:
        vsm_name = vsm.split('_percentile')[0]
        df[f'wgr_{vsm_name}'] = (
            df.groupby('ensg')[vsm]
              .rank(method='average', ascending=True)
        )

    df[f'wgr_{feature}'] = (
        df.groupby('ensg')[feature]
          .rank(method='average', ascending=True)
    )

    return df

def compute_gene_vsm_feature_corr(df_gene, vsms, feature, bool_feat):
    rows = []

    for vsm in vsms:
        vsm_name = vsm.split('_percentile')[0]
        x_col = f'wgr_{vsm_name}'
        if bool_feat:
            y_col=feature
        else:
            y_col = f'wgr_{feature}'

        df2 = df_gene[[x_col, y_col]].dropna()

        if len(df2) <= 1:
            return None

        X_raw = df2[[x_col]]
        if bool_feat:
            y_raw = df2[y_col].astype(bool)
        else:
            y_raw = df2[y_col]
        
        # ---- correlation ----
        corr = df2[x_col].corr(df2[y_col])

        rows.append({
            'vsm': vsm_name,
            'feature': feature,
            'corr': corr,
        })

    return rows

def compute_all_gene_results(df_clf, vsms, f):
    results = []

    genes = df_clf.ensg.unique()
    print(f'Genes to complete: {len(genes)}')

    for i, gene in enumerate(genes):
        df_gene = df_clf[df_clf.ensg == gene]

        rows = compute_gene_vsm_feature_corr(df_gene, vsms, f, False)

        if rows is not None:
            for r in rows:
                r['gene'] = gene
                results.append(r)
                
    return pd.DataFrame(results)

def compute_all_gene_results_boolean(df_clf, vsms, f):
    results = []

    genes = df_clf.ensg.unique()
    print(f'Genes to complete: {len(genes)}')

    for i, gene in enumerate(genes):
        df_gene = df_clf[df_clf.ensg == gene]
        if len(df_gene[f].unique()) >1:
            rows = compute_gene_vsm_feature_corr(df_gene, vsms, f, True)
        else:
            rows = None

        if rows is not None:
            for r in rows:
                r['gene'] = gene
                results.append(r)

    return pd.DataFrame(results)

def run_feature(df_c, f, vsms):
    df_cf = df_c[df_c[f].notna()]
    df_cf = add_within_gene_ranks(df_cf, vsms, f)
    results_df = compute_all_gene_results(df_cf, vsms, f)
    
    results_df.to_csv(
        f'all_stats/stats_vsm_{f}_genes_chr{chrom}.tsv',
        sep='\t',
        index=False
    )
    
def run_feature_boolean(df_c, f, vsms):  
    df_cf = df_c[df_c[f].notna()]
    
    # only want within_gene_rank for vsms, not for boolean feature
    for vsm in vsms:
        vsm_name = vsm.split('_percentile')[0]
        df_cf[f'wgr_{vsm_name}'] = (
            df_cf.groupby('ensg')[vsm]
            .rank(method='average', ascending=True)
        )
    results_df = compute_all_gene_results_boolean(df_cf, vsms, f)
    
    results_df.to_csv(
        f'all_stats/stats_vsm_{f}_genes_chr{chrom}.tsv',
        sep='\t',
        index=False
    )

In [4]:
chromosomes = [str(i) for i in range(1,23)] + ['X']
vsms = ['AM_score_percentile',
       'score_PAI3D_percentile', 'cadd_score_percentile',
       'cpt_score_percentile', 'gpn_msa_score_percentile',
       'MisFit_D_score_percentile', 'MisFit_S_score_percentile',
       'mpc_score_percentile', 'polyphen_score_percentile',
       'ESM_1v_percentile', 'EVE_percentile', 'popEVE_percentile',
       'mcap_score_percentile', 'esm1b_percentile', 'gmvp_score_percentile',
       'phylop_percentile', 'sift_score_percentile']
features = {'pLDDT': 'plddt', 'foldX': 'foldx_ddg', 'rSASA': 'rsasa', 'Mutation Rate': 'mu_snp',
           'MobiDB': 'mobidb_score'}
g2p_f_categorical = {'Secondary structure (DSSP 3-state)*': 
                     ['C (loop/coil)', 'H (helix)', 'B (undefined)']}
g2p_quantitative = {'Molar mass (g/mol)': 'molar_mass', 
                    'Hydropathy': 'hydropathy', 
                    'fpocket: druggability score*': 'druggability_score'}
g2p_f_binary = {'Compositional bias (UniProt)': 'compositional_bias', # would expect these variants to be benign
                'Cross-link (UniProt)': 'cross_link',
                'Disulfide bond (UniProt)': 'disulfide_bond', 
                'Domain (UniProt)': 'domain',
                'Glycosylation (UniProt)': 'glycosylation', 
                'Lipidation (UniProt)': 'lipidation', 
                'Motif (UniProt)': 'motif', 
                'Region (UniProt)': 'region',
                'Repeat (UniProt)': 'repeat',
                'Transmembrane (UniProt)': 'transmembrane',
                'Zinc finger (UniProt)': 'zinc_finger',
                'Acetylation': 'acetylation',
                'Methylation': 'methylation', 
                'Phosphorylation': 'phosphorylation', 
                'SUMOylation': 'SUMOylation', 
                'Ubiquitination': 'ubiquitination',
                'Intra-chain Hydrogen bond (AlphaFold2)': 'intra_chain_hydrogen_bond',
                'Intra-chain Non-bonded interaction (AlphaFold2)': 'intra_chain_nonbonded_interaction',
                'Intra-chain Salt bridge (AlphaFold2)': 'intra_chain_salt_bridge',
                'Signal (UniProt)': 'signal', 
                'Initiator methionine (UniProt)': 'initiator_methionine', 
                'Propeptide (UniProt)': 'propeptide', 
                'Peptide (UniProt)': 'peptide', 
                'Chain (UniProt)': 'chain', 
                'Topological domain (UniProt)': 'topological_domain', 
                'Intramembrane (UniProt)': 'intramembrane', 
                'Transit peptide (UniProt)': 'transit_peptide'}
g2p_change = {'Molar mass (g/mol)': 'delta_molar_mass', 
              'Hydropathy': 'delta_hydropathy'}
g2p_sites = ['Active site (UniProt)', 'Binding site (UniProt)', 'Site (UniProt)']

In [5]:
# vsm data - with ensg and on inner join
df = pd.read_parquet('gs://grohlicek/genetics_gym_vsm_all_content/vsm_all_2026_04_21/vsm_ensg_inner_pre.parquet')
# drop evaluation data columns - just want variant, ensg, and scores
df = df[['chrom', 'pos', 'ref', 'alt', 'AM_score_percentile',
       'score_PAI3D_percentile', 'cadd_score_percentile',
       'cpt_score_percentile', 'gpn_msa_score_percentile',
       'MisFit_D_score_percentile', 'MisFit_S_score_percentile',
       'mpc_score_percentile', 'polyphen_score_percentile',
       'ESM_1v_percentile', 'EVE_percentile', 'popEVE_percentile',
       'mcap_score_percentile', 'esm1b_percentile', 'gmvp_score_percentile',
       'phylop_percentile', 'sift_score_percentile', 'ensg']]
# data choices to debate
## remove where ensg is na
df = df[df.ensg.notna()]
## remove duplicates
df = df.drop_duplicates()
## clean up index
df = df.reset_index(drop=True)

In [6]:
# results directory
os.makedirs('all_stats', exist_ok=True)

In [7]:
for chrom in chromosomes:
    print(f'Chromosome {chrom}')

    # vsm scores for given chromosome
    df_c = df[df.chrom == 'chr' + chrom]
    # add mechanisms
    mech_df = pd.read_csv(f'../mechanism_table/mech_file_by_chrom/linker_mech_chrom_{chrom}.tsv.gz', sep='\t', compression='gzip')
    df_c = df_c.merge(mech_df, how='inner', on=['chrom', 'pos', 'ref', 'alt', 'ensg'])
    
    # run each feature
    print('starting features: foldx, mu, rsasa, plddt, mobidb, af2bind', 'delta_hydropathy', 'delta_molar_mass', 'druggability_score')
    for f in ['foldx_ddg', 'mu_snp', 'rsasa', 'plddt', 'mobidb_score', 'delta_hydropathy', 'delta_molar_mass', 'druggability_score']:
        run_feature(df_c, f, vsms)
    f = 'af2bind_pocket'
    run_feature_boolean(df_c, f, vsms)

    print('starting amino acid properties')
    for f in ['aromatic', 'charged', 'polar_neutral', 'nonpolar_neutral']:
        run_feature_boolean(df_c, f, vsms)

    print('starting g2p features')
    for f in g2p_f_binary.values():
        run_feature_boolean(df_c, f, vsms)
    ss_3 = ['ss_loop_coil', 'ss_helix', 'ss_undefined']
    for ss in ss_3:
        run_feature_boolean(df_c, ss, vsms)
    run_feature_boolean(df_c, 'all_uniprot_sites', vsms)

Chromosome 1


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_91893/660340525.py:7: DtypeWarning: Columns (0: af2bind_pocket, 1: compositional_bias, 2: cross_link, 3: disulfide_bond, 4: domain, 5: glycosylation, 6: lipidation, 7: motif, 8: region, 9: repeat, 10: transmembrane, 11: zinc_finger, 12: acetylation, 13: methylation, 14: phosphorylation, 15: SUMOylation, 16: ubiquitination, 17: intra_chain_hydrogen_bond, 18: intra_chain_nonbonded_interaction, 19: intra_chain_salt_bridge, 20: signal, 21: initiator_methionine, 22: propeptide, 23: peptide, 24: chain, 25: topological_domain, 26: intramembrane, 27: transit_peptide, 28: ss_loop_coil, 29: ss_helix, 30: ss_undefined, 31: all_uniprot_sites) have mixed types. Specify dtype option on import or set low_memory=False.
  mech_df = pd.read_csv(f'../mechanism_table/mech_file_by_chrom/linker_mech_chrom_{chrom}.tsv.gz', sep='\t', compression='gzip')


starting features: foldx, mu, rsasa, plddt, mobidb, af2bind delta_hydropathy delta_molar_mass druggability_score
Genes to complete: 1466


/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Genes to complete: 1510


...

starting features: foldx, mu, rsasa, plddt, mobidb, af2bind delta_hydropathy delta_molar_mass druggability_score
Genes to complete: 308
Genes to complete: 322
Genes to complete: 308
Genes to complete: 308
Genes to complete: 181
Genes to complete: 322
Genes to complete: 322
Genes to complete: 308
Genes to complete: 203
starting amino acid properties
Genes to complete: 322
Genes to complete: 322
Genes to complete: 322
Genes to complete: 322
starting g2p features
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes 

/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_91893/660340525.py:7: DtypeWarning: Columns (0: af2bind_pocket, 1: compositional_bias, 2: cross_link, 3: disulfide_bond, 4: domain, 5: glycosylation, 6: lipidation, 7: motif, 8: region, 9: repeat, 10: transmembrane, 11: zinc_finger, 12: acetylation, 13: methylation, 14: phosphorylation, 15: SUMOylation, 16: ubiquitination, 17: intra_chain_hydrogen_bond, 18: intra_chain_nonbonded_interaction, 19: intra_chain_salt_bridge, 20: signal, 21: initiator_methionine, 22: propeptide, 23: peptide, 24: chain, 25: topological_domain, 26: intramembrane, 27: transit_peptide, 28: ss_loop_coil, 29: ss_helix, 30: ss_undefined, 31: all_uniprot_sites) have mixed types. Specify dtype option on import or set low_memory=False.
  mech_df = pd.read_csv(f'../mechanism_table/mech_file_by_chrom/linker_mech_chrom_{chrom}.tsv.gz', sep='\t', compression='gzip')


starting features: foldx, mu, rsasa, plddt, mobidb, af2bind delta_hydropathy delta_molar_mass druggability_score
Genes to complete: 480
Genes to complete: 491
Genes to complete: 480
Genes to complete: 480
Genes to complete: 308
Genes to complete: 491
Genes to complete: 491
Genes to complete: 480


/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, N

Genes to complete: 300
starting amino acid properties
Genes to complete: 491
Genes to complete: 491
Genes to complete: 491
Genes to complete: 491
starting g2p features
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
